# Country Performance - Business Metrics

Analyzes sales performance by country/geography. Answers:
- Which countries generate the most revenue?
- How many unique customers per country?
- What's the average order value by country?
- Holiday impact by country

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.agg_country_performance AS
SELECT 
  cu.country,
  COUNT(DISTINCT fe.order_number) AS total_orders,
  COUNT(DISTINCT cu.customer_id) AS unique_customers,
  SUM(fe.sales_amount) AS total_revenue,
  ROUND(AVG(fe.sales_amount), 2) AS avg_order_value,
  SUM(fe.quantity) AS total_quantity,
  
  -- Holiday impact
  SUM(CASE WHEN fe.is_holiday THEN 1 ELSE 0 END) AS holiday_orders,
  ROUND(100.0 * SUM(CASE WHEN fe.is_holiday THEN 1 ELSE 0 END) / COUNT(*), 2) AS holiday_order_pct,
  SUM(CASE WHEN fe.is_holiday THEN fe.sales_amount ELSE 0 END) AS holiday_revenue

FROM workspace.gold.fact_sales_enriched fe
INNER JOIN workspace.gold.dim_customers cu 
  ON fe.customer_key = cu.customer_key
  AND cu.is_current = TRUE

WHERE fe.order_date IS NOT NULL

GROUP BY cu.country
ORDER BY total_revenue DESC

In [0]:
%sql
SELECT 
  country,
  total_orders,
  unique_customers,
  total_revenue,
  avg_order_value,
  holiday_order_pct
FROM workspace.gold.agg_country_performance
ORDER BY total_revenue DESC

In [0]:
%sql
SELECT 
  COUNT(DISTINCT country) AS total_countries,
  SUM(total_revenue) AS global_revenue,
  SUM(unique_customers) AS total_customers,
  ROUND(AVG(avg_order_value), 2) AS avg_order_value_global,
  ROUND(AVG(holiday_order_pct), 2) AS avg_holiday_pct
FROM workspace.gold.agg_country_performance